# T4 · `orderbook_snapshots`: Order book snapshots

Full depth for flagged markets, one row per price level. One file per platform per book tick.

See `DECISIONS.md` for the full column definitions. Re-generate with `python -m collector display`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

DATA = Path.cwd().parent                 # this notebook lives in data/display/
sys.path.insert(0, str(DATA.parent))     # repo root, for collector.schemas
from collector import schemas

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)

## Build the table: concatenate its parquet files

In [2]:
files = sorted(DATA.glob("orderbook_snapshots/*/*/*.parquet"))
print(f"{len(files)} file(s)")

if files:
    # numpy_nullable keeps integer timestamps as integers even when some are missing
    df = pd.concat([pd.read_parquet(f, dtype_backend="numpy_nullable") for f in files], ignore_index=True)
else:  # table not populated yet: show its columns
    df = schemas.ORDERBOOK_SNAPSHOTS.empty_table().to_pandas()

print("shape:", df.shape)

0 file(s)
shape: (0, 14)


## Columns

In [3]:
print(f"{len(df.columns)} columns:")
for i, (col, dtype) in enumerate(df.dtypes.items(), 1):
    print(f"{i:3d}. {col:28s} {dtype}")

14 columns:
  1. tick_ts                      int64
  2. platform                     object
  3. market_uid                   object
  4. request_sent_ts              int64
  5. response_received_ts         int64
  6. exchange_ts                  int64
  7. outcome                      object
  8. side                         object
  9. level                        int64
 10. price_c                      float64
 11. qty                          float64
 12. derived                      bool
 13. book_hash                    object
 14. book_state                   object


## View: 15 rows per platform

In [4]:
if df.empty:
    print("No rows yet.")
    display(df)
elif "platform" in df.columns:
    for platform, g in df.groupby("platform"):
        print(f"{platform}: {len(g):,} rows (showing 15)")
        display(g.head(15))
else:
    display(df.head(15))

No rows yet.


,tick_ts,platform,market_uid,request_sent_ts,response_received_ts,exchange_ts,outcome,side,level,price_c,qty,derived,book_hash,book_state


## Example: one live order book per platform

T4 only fills once pairs are flagged in `match_flags`. To show what it will look like, this cell picks
one liquid open market on each platform, pulls its order book **at notebook build time**, and converts it
with the collector's own code (`kalshi.book_rows` / `polyus.book_rows`), so the rows have exactly the T4 format.
Nothing here is written to `data/`.

- Kalshi returns bids only; YES asks are derived as 100 - NO bid (`derived = True`).
- Polymarket US returns YES bids/offers; the NO side is derived as 100 - YES.

In [5]:
import httpx
from collector import kalshi, polyus
from collector.config import KALSHI_BASE, POLYUS_BASE, USER_AGENT
from collector.http import Resp
from collector.util import now_ms

def latest_pricing(platform, cols):
    """Pricing rows are change-only: the latest state of a market is its last row."""
    fs = sorted(DATA.glob(f"pricing_snapshots/{platform}/*/*.parquet"))
    px = pd.concat([pd.read_parquet(f, columns=cols) for f in fs], ignore_index=True)
    return px.sort_values("tick_ts").groupby("market_uid").tail(1)

cols = ["tick_ts", "market_uid", "status_canonical", "yes_bid_c", "yes_ask_c"]

def mid_priced(df):
    """Open, two-sided quote, YES bid between 20c and 80c (a more interesting book than a 2c longshot)."""
    return df[(df.status_canonical == "open") & df.yes_bid_c.between(20, 80) & (df.yes_ask_c < 100)]

# Kalshi: mid-priced open market with the highest 24h volume
k = mid_priced(latest_pricing("KALSHI", cols + ["volume_24h"]))
k_uid = k.sort_values("volume_24h", ascending=False).market_uid.iloc[0]

# Polymarket US: mid-priced open market with the most shares traded (T6), else the tightest spread
p = mid_priced(latest_pricing("POLYUS", cols))
sfs = sorted(DATA.glob("market_stats/POLYUS/*/*.parquet"))
st = pd.concat([pd.read_parquet(f, columns=["market_uid", "shares_traded"]) for f in sfs]) if sfs else pd.DataFrame()
st = st[st.market_uid.isin(p.market_uid)] if len(st) else st
if len(st):
    p_uid = st.sort_values("shares_traded", ascending=False).market_uid.iloc[0]
else:
    p_uid = p.assign(spread=p.yes_ask_c - p.yes_bid_c).sort_values("spread").market_uid.iloc[0]

catalog = pd.concat([pd.read_parquet(DATA / f"market_catalog/current/{x}.parquet",
                                     columns=["market_uid", "event_title", "market_title", "market_question",
                                              "yes_title", "yes_side_label", "market_slug"])
                     for x in ("KALSHI", "POLYUS")])
display(catalog[catalog.market_uid.isin([k_uid, p_uid])])

,market_uid,event_title,market_title,market_question,yes_title,yes_side_label,market_slug
175123,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,Jacksonville vs Baltimore,Baltimore wins,Baltimore wins,Baltimore,Yes,None
266,POLYUS:65588,Anthropic IPO Officially Confirmed?,"November 30, 2026",Anthropic IPO Officially Confirmed By,Anthropic IPO Officially Confirmed By,Yes,ipcc-anthropic-2026-11-30


In [6]:
def fetch(base, path):
    sent = now_ms()
    r = httpx.get(base + path, timeout=30, headers={"User-Agent": USER_AGENT})
    recv = now_ms()
    r.raise_for_status()
    return Resp(r.json(), r.status_code, sent, recv, len(r.content))

tick = (now_ms() // 1000) * 1000
kr = fetch(KALSHI_BASE, f"/markets/{k_uid.split(':', 1)[1]}/orderbook")
k_rows, _ = kalshi.book_rows(k_uid, kr.data, tick, kr)

slug = catalog.set_index("market_uid").market_slug[p_uid]
pr = fetch(POLYUS_BASE, f"/v1/markets/{slug}/book")
p_rows, _, _ = polyus.book_rows(p_uid, pr.data, tick, pr)

import pyarrow as pa
example = (pa.Table.from_pylist(k_rows + p_rows, schema=schemas.ORDERBOOK_SNAPSHOTS)
           .to_pandas(types_mapper={pa.int64(): pd.Int64Dtype()}.get))
order = {"yes": 0, "no": 1, "bid": 0, "ask": 1}
example = example.sort_values(["platform", "outcome", "side", "level"],
                              key=lambda s: s.map(order) if s.name in ("outcome", "side") else s,
                              ignore_index=True)
print("pulled at", pd.to_datetime(tick, unit="ms", utc=True))
for platform, g in example.groupby("platform"):
    print(f"{platform} {g.market_uid.iloc[0]}: {len(g)} level rows "
          f"(showing top 5 levels of each outcome/side)")
    display(g.groupby(["outcome", "side"], sort=False).head(5))

pulled at 2026-11-06 00:35:13+00:00
KALSHI KALSHI:KXNFLGAME-26NOV05JACBAL-BAL: 198 level rows (showing top 5 levels of each outcome/side)


,tick_ts,platform,market_uid,request_sent_ts,response_received_ts,exchange_ts,outcome,side,level,price_c,qty,derived,book_hash,book_state
0,1793925313000,KALSHI,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,1793925313415,1793925313692,<NA>,yes,bid,0,21.0,1219798.76,False,03d63be4981d6ec1c0dfb7a7792f752ed8336b21,None
1,1793925313000,KALSHI,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,1793925313415,1793925313692,<NA>,yes,bid,1,20.0,1768214.58,False,03d63be4981d6ec1c0dfb7a7792f752ed8336b21,None
2,1793925313000,KALSHI,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,1793925313415,1793925313692,<NA>,yes,bid,2,19.0,541203.64,False,03d63be4981d6ec1c0dfb7a7792f752ed8336b21,None
3,1793925313000,KALSHI,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,1793925313415,1793925313692,<NA>,yes,bid,3,18.0,104209.83,False,03d63be4981d6ec1c0dfb7a7792f752ed8336b21,None
4,1793925313000,KALSHI,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,1793925313415,1793925313692,<NA>,yes,bid,4,17.0,24835.67,False,03d63be4981d6ec1c0dfb7a7792f752ed8336b21,None
21,1793925313000,KALSHI,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,1793925313415,1793925313692,<NA>,yes,ask,0,22.0,3509337.69,True,03d63be4981d6ec1c0dfb7a7792f752ed8336b21,None
22,1793925313000,KALSHI,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,1793925313415,1793925313692,<NA>,yes,ask,1,23.0,3485700.74,True,03d63be4981d6ec1c0dfb7a7792f752ed8336b21,None
23,1793925313000,KALSHI,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,1793925313415,1793925313692,<NA>,yes,ask,2,24.0,2647697.46,True,03d63be4981d6ec1c0dfb7a7792f752ed8336b21,None
24,1793925313000,KALSHI,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,1793925313415,1793925313692,<NA>,yes,ask,3,25.0,114979.10,True,03d63be4981d6ec1c0dfb7a7792f752ed8336b21,None
25,1793925313000,KALSHI,KALSHI:KXNFLGAME-26NOV05JACBAL-BAL,1793925313415,1793925313692,<NA>,yes,ask,4,26.0,10144.96,True,03d63be4981d6ec1c0dfb7a7792f752ed8336b21,None


POLYUS POLYUS:65588: 60 level rows (showing top 5 levels of each outcome/side)


,tick_ts,platform,market_uid,request_sent_ts,response_received_ts,exchange_ts,outcome,side,level,price_c,qty,derived,book_hash,book_state
198,1793925313000,POLYUS,POLYUS:65588,1793925313738,1793925313784,1793925296076,yes,bid,0,52.0,2.0,False,7b52ce0c44823d191fcabac7755445f14880ab05,MARKET_STATE_OPEN
199,1793925313000,POLYUS,POLYUS:65588,1793925313738,1793925313784,1793925296076,yes,bid,1,49.0,1.0,False,7b52ce0c44823d191fcabac7755445f14880ab05,MARKET_STATE_OPEN
200,1793925313000,POLYUS,POLYUS:65588,1793925313738,1793925313784,1793925296076,yes,bid,2,40.0,50.0,False,7b52ce0c44823d191fcabac7755445f14880ab05,MARKET_STATE_OPEN
201,1793925313000,POLYUS,POLYUS:65588,1793925313738,1793925313784,1793925296076,yes,bid,3,39.0,5.0,False,7b52ce0c44823d191fcabac7755445f14880ab05,MARKET_STATE_OPEN
202,1793925313000,POLYUS,POLYUS:65588,1793925313738,1793925313784,1793925296076,yes,bid,4,38.0,301.0,False,7b52ce0c44823d191fcabac7755445f14880ab05,MARKET_STATE_OPEN
212,1793925313000,POLYUS,POLYUS:65588,1793925313738,1793925313784,1793925296076,yes,ask,0,53.0,920.0,False,7b52ce0c44823d191fcabac7755445f14880ab05,MARKET_STATE_OPEN
213,1793925313000,POLYUS,POLYUS:65588,1793925313738,1793925313784,1793925296076,yes,ask,1,54.0,127.0,False,7b52ce0c44823d191fcabac7755445f14880ab05,MARKET_STATE_OPEN
214,1793925313000,POLYUS,POLYUS:65588,1793925313738,1793925313784,1793925296076,yes,ask,2,55.0,1000.0,False,7b52ce0c44823d191fcabac7755445f14880ab05,MARKET_STATE_OPEN
215,1793925313000,POLYUS,POLYUS:65588,1793925313738,1793925313784,1793925296076,yes,ask,3,57.0,250.0,False,7b52ce0c44823d191fcabac7755445f14880ab05,MARKET_STATE_OPEN
216,1793925313000,POLYUS,POLYUS:65588,1793925313738,1793925313784,1793925296076,yes,ask,4,60.0,246.0,False,7b52ce0c44823d191fcabac7755445f14880ab05,MARKET_STATE_OPEN


In [7]:
# Same data as a YES-side price ladder (best 10 levels): bids on the left, asks on the right
for platform, g in example.groupby("platform"):
    y = g[g.outcome == "yes"]
    bid = y[y.side == "bid"].set_index("level")[["qty", "price_c"]].add_prefix("bid_")
    ask = y[y.side == "ask"].set_index("level")[["price_c", "qty"]].add_prefix("ask_")
    print(platform, g.market_uid.iloc[0])
    display(bid.join(ask, how="outer").head(10))

KALSHI KALSHI:KXNFLGAME-26NOV05JACBAL-BAL


,bid_qty,bid_price_c,ask_price_c,ask_qty
level,,,,
0,1219798.76,21.0,22.0,3509337.69
1,1768214.58,20.0,23.0,3485700.74
2,541203.64,19.0,24.0,2647697.46
3,104209.83,18.0,25.0,114979.10
4,24835.67,17.0,26.0,10144.96
5,12116.59,16.0,27.0,1334.61
6,1373.75,15.0,28.0,12149.27
7,789.55,14.0,29.0,5036.29
8,338.32,13.0,30.0,682.20


POLYUS POLYUS:65588


,bid_qty,bid_price_c,ask_price_c,ask_qty
level,,,,
0,2.0,52.0,53.0,920.0
1,1.0,49.0,54.0,127.0
2,50.0,40.0,55.0,1000.0
3,5.0,39.0,57.0,250.0
4,301.0,38.0,60.0,246.0
5,556.0,37.0,61.0,111.0
6,1027.0,36.0,62.0,76.0
7,300.0,35.0,63.0,1091.0
8,500.0,30.0,69.0,118.0
